# Burst-by-burst Doppler comparison: SLC pipeline vs OCN product

Compares our CDE-estimated raw Doppler centroid (`doppler_obs`) with the
OCN product's `rvlDcObs` field on a burst-by-burst basis.

**Variables compared**
| Field | Source | Units |
|---|---|---|
| `doppler_obs` | `burst_pipeline.compute_rvl_burst` | Hz |
| `rvlDcObs` | Level-2 OCN SAFE | Hz |

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from scripts.sentinel_1.doppler_comparison import (
    compare_burst_doppler,
    compare_all_bursts,
    burst_summary_stats,
    load_ocn_rvl_swath,
    assign_burst_indices,
)
from scripts.sentinel_1.safe_io import find_safe_files, parse_annotation

import os

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## Paths and parameters

In [ ]:
ROOT        = os.path.dirname(os.getcwd())          # repo root
DATA        = os.path.join(ROOT, 'ocean_current_retrieval/data')
SCENE       = f"scene{1}"

SLC_SAFE    = os.path.join(DATA, 'sentinel-1', SCENE,
    'S1A_IW_SLC.SAFE')
OCN_NC     = os.path.join(DATA, 'sentinel-1', SCENE, 'S1A_IW_OCN.SAFE', 'measurement', 's1a-iw-ocn-vv-20260205t165251-20260205t165322-063086-07EAEF-001.nc')
SUBSWATH   = 'iw1'
POL        = 'vv'

# CDE block parameters
BLOCK_AZ   = 256
BLOCK_RG   = 512
STRIDE_AZ  = 128
STRIDE_RG  = 256

## Overview: burst layout in the OCN product

In [ ]:
files = find_safe_files(SLC_SAFE, SUBSWATH, POL)
annot = parse_annotation(files['annotation'])
swath_idx = int(SUBSWATH[-1]) - 1

ocn_ds = load_ocn_rvl_swath(OCN_NC, swath_idx)
burst_assign = assign_burst_indices(OCN_NC, annot, swath_idx)

print(f'Subswath : {SUBSWATH.upper()}')
print(f'SLC bursts: {len(annot.bursts)}')
print(f'OCN rows  : {len(burst_assign)}')
print()
for bi in range(len(annot.bursts)):
    n = (burst_assign == bi).sum()
    t = annot.bursts[bi].azimuth_time.strftime('%H:%M:%S.%f')
    print(f'  Burst {bi}: {n:3d} OCN rows   az_time={t}')

## Single-burst deep dive

In [ ]:
BURST_IDX = 4   # change to inspect a different burst

cmp = compare_burst_doppler(
    SLC_SAFE, OCN_NC, SUBSWATH, BURST_IDX, POL,
    block_az=BLOCK_AZ, block_rg=BLOCK_RG,
    stride_az=STRIDE_AZ, stride_rg=STRIDE_RG,
)

our   = cmp['our']
ocn_b = cmp['ocn']
diff  = cmp['diff']
our_rg = cmp['our_regrid']

ocn_dc = ocn_b['rvlDcObs'].values.astype(float)
valid  = np.isfinite(diff) & np.isfinite(ocn_dc)

print(f'Burst {BURST_IDX}  |  valid comparison cells: {valid.sum()}/{diff.size}')
if valid.sum() > 0:
    d = diff[valid]
    print(f'  bias = {d.mean():.2f} Hz')
    print(f'  std  = {d.std():.2f} Hz')
    print(f'  RMSE = {np.sqrt((d**2).mean()):.2f} Hz')

In [ ]:
# ── side-by-side images on each dataset's own block / OCN grid ──────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
fig.suptitle(f'{SUBSWATH.upper()} burst {BURST_IDX} — raw Doppler centroid [Hz]')

obs_vals  = our['doppler_obs'].values
vmax_obs  = np.nanpercentile(np.abs(obs_vals), 98)
norm_obs  = mcolors.TwoSlopeNorm(vcenter=0, vmin=-vmax_obs, vmax=vmax_obs)

im0 = axes[0].imshow(obs_vals, aspect='auto', cmap='RdBu_r', norm=norm_obs)
axes[0].set_title('Our  doppler_obs\n(CDE block grid)')
axes[0].set_xlabel('Range cell'); axes[0].set_ylabel('Azimuth cell')
plt.colorbar(im0, ax=axes[0], label='Hz')

ocn_vals = ocn_dc
vmax_ocn = np.nanpercentile(np.abs(ocn_vals[np.isfinite(ocn_vals)]), 98) if np.isfinite(ocn_vals).any() else 1
norm_ocn = mcolors.TwoSlopeNorm(vcenter=0, vmin=-vmax_ocn, vmax=vmax_ocn)

im1 = axes[1].imshow(ocn_vals, aspect='auto', cmap='RdBu_r', norm=norm_ocn)
axes[1].set_title('OCN  rvlDcObs\n(OCN grid)')
axes[1].set_xlabel('Range cell'); axes[1].set_ylabel('Azimuth row (burst)')
plt.colorbar(im1, ax=axes[1], label='Hz')

diff_vals = diff
vmax_d    = np.nanpercentile(np.abs(diff_vals[np.isfinite(diff_vals)]), 98) if np.isfinite(diff_vals).any() else 1
norm_d    = mcolors.TwoSlopeNorm(vcenter=0, vmin=-vmax_d, vmax=vmax_d)

im2 = axes[2].imshow(diff_vals, aspect='auto', cmap='RdBu_r', norm=norm_d)
axes[2].set_title('Difference\n(our − OCN, regridded)')
axes[2].set_xlabel('Range cell'); axes[2].set_ylabel('Azimuth row (burst)')
plt.colorbar(im2, ax=axes[2], label='Hz')

plt.tight_layout()
plt.show()

In [ ]:
# ── scatter and histogram ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle(f'{SUBSWATH.upper()} burst {BURST_IDX} — Doppler comparison')

if valid.sum() > 0:
    x = ocn_dc[valid]
    y = our_rg[valid]

    lo = min(np.nanpercentile(x, 1), np.nanpercentile(y, 1))
    hi = max(np.nanpercentile(x, 99), np.nanpercentile(y, 99))

    axes[0].scatter(x, y, s=1)#, gridsize=50, cmap='viridis', mincnt=1)
    axes[0].plot([lo, hi], [lo, hi], 'r--', lw=1.5, label='1:1')
    axes[0].set_xlabel('OCN rvlDcObs [Hz]')
    axes[0].set_ylabel('Our doppler_obs (regridded) [Hz]')
    axes[0].set_title('Scatter')
    axes[0].set_xlim(lo, hi); axes[0].set_ylim(lo, hi)
    axes[0].legend()

    d = diff[valid]
    axes[1].hist(d, bins=60, color='steelblue', edgecolor='none', alpha=0.8)
    axes[1].axvline(d.mean(), color='red', lw=2, label=f'bias={d.mean():.2f} Hz')
    axes[1].axvline(0, color='black', lw=1, ls='--')
    axes[1].set_xlabel('Our − OCN [Hz]')
    axes[1].set_ylabel('Count')
    axes[1].set_title(f'Difference histogram  (σ={d.std():.2f} Hz)')
    axes[1].legend()
else:
    for ax in axes:
        ax.text(0.5, 0.5, 'No valid overlap', ha='center', va='center', transform=ax.transAxes)

plt.tight_layout()
plt.show()

In [ ]:
from scipy.interpolate import griddata

def _to_grid(lon, lat, values, n=200):
    """Interpolate scattered (lon, lat, values) onto a regular n×n grid."""
    lon_g = np.linspace(np.nanmin(lon), np.nanmax(lon), n)
    lat_g = np.linspace(np.nanmin(lat), np.nanmax(lat), n)
    lon_grid, lat_grid = np.meshgrid(lon_g, lat_g)
    pts   = np.column_stack([lon.ravel(), lat.ravel()])
    vals  = values.ravel()
    ok    = np.isfinite(vals)
    grid  = griddata(pts[ok], vals[ok], (lon_grid, lat_grid), method='linear')
    return lon_g, lat_g, grid

# ── geographic view ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
fig.suptitle(f'{SUBSWATH.upper()} burst {BURST_IDX} — geographic view')

our_lat = our['latitude'].values
our_lon = our['longitude'].values
ocn_lat = ocn_b['rvlLat'].values
ocn_lon = ocn_b['rvlLon'].values

# Our doppler_obs
lon_g, lat_g, grid0 = _to_grid(our_lon, our_lat, obs_vals)
im0 = axes[0].pcolormesh(lon_g, lat_g, grid0, cmap='RdBu_r', norm=norm_obs, shading='auto')
axes[0].set_title('Our doppler_obs [Hz]')
axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')
plt.colorbar(im0, ax=axes[0], label='Hz')

# OCN rvlDcObs
lon_g, lat_g, grid1 = _to_grid(ocn_lon, ocn_lat, ocn_dc)
im1 = axes[1].pcolormesh(lon_g, lat_g, grid1, cmap='RdBu_r', norm=norm_ocn, shading='auto')
axes[1].set_title('OCN rvlDcObs [Hz]')
axes[1].set_xlabel('Longitude')
plt.colorbar(im1, ax=axes[1], label='Hz')

# Difference
lon_g, lat_g, grid2 = _to_grid(ocn_lon, ocn_lat, diff_vals)
im2 = axes[2].pcolormesh(lon_g, lat_g, grid2, cmap='RdBu_r', norm=norm_d, shading='auto')
axes[2].set_title('Difference (our − OCN) [Hz]')
axes[2].set_xlabel('Longitude')
plt.colorbar(im2, ax=axes[2], label='Hz')

plt.tight_layout()
plt.show()

## All-burst comparison

In [ ]:
print(f'Processing all {len(annot.bursts)} bursts of {SUBSWATH.upper()} …')
comparisons = compare_all_bursts(
    SLC_SAFE, OCN_NC, SUBSWATH, POL,
    block_az=BLOCK_AZ, block_rg=BLOCK_RG,
    stride_az=STRIDE_AZ, stride_rg=STRIDE_RG,
)
stats = burst_summary_stats(comparisons)
print('Done.')

In [ ]:
import pandas as pd

n_bursts = len(annot.bursts)
df = pd.DataFrame({
    'burst':        np.arange(n_bursts),
    'our_mean_hz':  stats['our_mean_hz'].round(2),
    'ocn_mean_hz':  stats['ocn_mean_hz'].round(2),
    'bias_hz':      stats['bias_hz'].round(2),
    'std_hz':       stats['std_hz'].round(2),
    'rmse_hz':      stats['rmse_hz'].round(2),
    'n_valid':      stats['n_valid'],
})
df.set_index('burst', inplace=True)
df

In [ ]:
burst_nums = np.arange(n_bursts)

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
fig.suptitle(f'{SUBSWATH.upper()} — per-burst Doppler comparison')

# Top: mean Doppler
axes[0].plot(burst_nums, stats['our_mean_hz'], 'o-', color='steelblue', label='Our (mean)')
axes[0].plot(burst_nums, stats['ocn_mean_hz'], 's--', color='tomato',   label='OCN (mean)')
axes[0].set_ylabel('Mean Doppler [Hz]')
axes[0].set_title('Mean raw Doppler per burst')
axes[0].legend()
axes[0].grid(True, alpha=0.4)

# Bottom: bias ± std
axes[1].axhline(0, color='black', lw=0.8, ls='--')
axes[1].bar(burst_nums, stats['bias_hz'], color='steelblue', alpha=0.7, label='bias (our − OCN)')
axes[1].errorbar(
    burst_nums, stats['bias_hz'],
    yerr=stats['std_hz'], fmt='none', color='navy', capsize=4, lw=1.5, label='±1σ'
)
axes[1].set_xlabel('Burst index')
axes[1].set_ylabel('Bias [Hz]')
axes[1].set_title('Bias and spread (our − OCN)')
axes[1].set_xticks(burst_nums)
axes[1].legend()
axes[1].grid(True, alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# ── three-panel gallery for all bursts: our | OCN | difference ──────────────
fig, axes = plt.subplots(n_bursts, 3, figsize=(15, n_bursts * 3))
fig.suptitle(f'{SUBSWATH.upper()} — per-burst Doppler [Hz]', y=1.01)

# shared colour scales
all_our   = np.concatenate([c['our']['doppler_obs'].values.ravel() for c in comparisons])
all_ocn   = np.concatenate([c['ocn']['rvlDcObs'].values.astype(float).ravel() for c in comparisons])
all_diffs = np.concatenate([c['diff'].ravel() for c in comparisons])

vmax_our  = np.nanpercentile(np.abs(all_our[np.isfinite(all_our)]), 98)   if np.isfinite(all_our).any()   else 1
vmax_ocn  = np.nanpercentile(np.abs(all_ocn[np.isfinite(all_ocn)]), 98)   if np.isfinite(all_ocn).any()   else 1
vmax_diff = np.nanpercentile(np.abs(all_diffs[np.isfinite(all_diffs)]), 98) if np.isfinite(all_diffs).any() else 1

norm_our  = mcolors.TwoSlopeNorm(vcenter=0, vmin=-vmax_our,  vmax=vmax_our)
norm_ocn  = mcolors.TwoSlopeNorm(vcenter=0, vmin=-vmax_ocn,  vmax=vmax_ocn)
norm_diff = mcolors.TwoSlopeNorm(vcenter=0, vmin=-vmax_diff, vmax=vmax_diff)

for bi, cmp in enumerate(comparisons):
    our_vals  = cmp['our']['doppler_obs'].values
    ocn_vals  = cmp['ocn']['rvlDcObs'].values.astype(float)
    diff_vals = cmp['diff']
    bias = np.nanmean(diff_vals)

    im0 = axes[bi, 0].imshow(our_vals,  aspect='auto', cmap='RdBu_r', norm=norm_our)
    im1 = axes[bi, 1].imshow(ocn_vals,  aspect='auto', cmap='RdBu_r', norm=norm_ocn)
    im2 = axes[bi, 2].imshow(diff_vals, aspect='auto', cmap='RdBu_r', norm=norm_diff)

    axes[bi, 0].set_ylabel(f'Burst {bi}', fontsize=9)
    axes[bi, 2].text(0.02, 0.97, f'bias={bias:+.1f} Hz',
                     transform=axes[bi, 2].transAxes,
                     fontsize=7, va='top',
                     bbox=dict(facecolor='white', alpha=0.6, pad=2))

    for ax in axes[bi]:
        ax.set_xticks([]); ax.set_yticks([])

axes[0, 0].set_title('Our doppler_obs')
axes[0, 1].set_title('OCN rvlDcObs')
axes[0, 2].set_title('Difference (our − OCN)')

fig.colorbar(im0, ax=axes[:, 0].tolist(), label='Hz', shrink=0.6)
fig.colorbar(im1, ax=axes[:, 1].tolist(), label='Hz', shrink=0.6)
fig.colorbar(im2, ax=axes[:, 2].tolist(), label='Hz', shrink=0.6)

plt.tight_layout()
plt.show()